In [78]:
import datasets
import json
from tqdm import tqdm
from functools import partial

In [79]:
seed = 2024
import torch
from transformers import AutoTokenizer

# model_id = "/data/models/gemma-2b"
# model_id = "/data/models/llama-2-7b"
# model_id = "/data/models/llama-3-8b"
model_id = "/data/models/Mistral-7B-v0.1"
truncation = True
model_max_length = 2048
tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=model_max_length, use_fast=True, truncation=truncation)
tokenizer.pad_token = tokenizer.unk_token
from conversation import get_conv_template
# conv = get_conv_template("gemma")
# conv = get_conv_template("llama-3")
conv = get_conv_template("vicuna_v1.1")

In [80]:
folder = "zip"
data_name = "zip_10000"
out_name = "zip_10000_truncated"

In [81]:
folder = "token_mistral_re"
tag = "shortest"
data_name = f"token_mistral_re_{tag}"
out_name = f"token_mistral_re_{tag}_truncated"

In [82]:
raw_dataset = json.load(open(f'{folder}/{data_name}.json'))

In [83]:
"""
1. remove extra keys in conversations
2. remove incorrect conversations, like [human, human, gpt]
"""
roles = ['human', 'gpt']
cleaned_dataset = []
print(len(raw_dataset))
for line in tqdm(raw_dataset):
    new_conversations = []
    flag = True
    tmp = line['conversations']
    if len(tmp) % 2 != 0:
        tmp = tmp[:-1]
    if len(tmp) == 0:
        continue
    for idx, _ in enumerate(line['conversations']):
        if _['from'] != roles[idx % 2]:
            flag = False
            break
        new_conversations.append({
            'from': _['from'],
            'value': _['value']
        })
    if flag:
        cleaned_dataset.append({
            'id': line['id'],
            'source': line['source'],
            'conversations': new_conversations,
        })
print(len(cleaned_dataset))
raw_dataset = cleaned_dataset
raw_dataset = datasets.Dataset.from_list(raw_dataset)

31039


100%|██████████| 31039/31039 [00:00<00:00, 152360.66it/s]


31039


In [84]:
def truncate_long_seq(sample):
    new_sample = {}
    roles = {"human": conv.roles[0], "gpt": conv.roles[1]}
    conv.messages = []
    end_turn_idx = 0
    flag = False
    for j, sentence in enumerate(sample['conversations']):
        role = roles[sentence["from"]]
        assert role == conv.roles[j % 2]
        conv.append_message(role, sentence["value"])
        if j % 2 == 1: # process one turn
            conversation = conv.get_prompt()
            cur_len = len(tokenizer(conversation, truncation=False).input_ids)
            if cur_len > model_max_length:
                flag = True
                if end_turn_idx == 0:
                    end_turn_idx = 2
                break
            else:
                end_turn_idx = j + 1
    new_sample = {
        'id': sample['id'],
        'source': sample['source'],
        'conversations': sample['conversations'][:end_turn_idx],
        'drop': flag,
    }
    return new_sample
truncate_long = partial(
    truncate_long_seq,
)
raw_dataset = raw_dataset.map(
    truncate_long,
    num_proc=64,
    desc="Truncating Long Sequences",
)



Truncating Long Sequences (num_proc=64): 100%|██████████| 31039/31039 [00:13<00:00, 2221.93 examples/s]


In [85]:
raw_dataset = raw_dataset.filter(
    lambda x: len(x['conversations']) > 0,
    num_proc=64,
    desc="Filter out empty Sequences after dropping long conversations",
)

Filter out empty Sequences after dropping long conversations (num_proc=64): 100%|██████████| 31039/31039 [00:00<00:00, 42241.94 examples/s]


In [86]:
raw_dataset

Dataset({
    features: ['id', 'source', 'conversations', 'drop'],
    num_rows: 31039
})

In [87]:
with open(f'{folder}/{out_name}.json', 'w+') as f:
    json.dump(raw_dataset.to_list(), f, indent=4)